In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

os.chdir('/content/drive/MyDrive/GNNs/HIV inhibitors-GNN')
print("Current directory:", os.getcwd())

Current directory: /content/drive/MyDrive/GNNs/HIV inhibitors-GNN


In [ ]:
import pandas as pd

data_path = 'data/raw/HIV.csv'

data = pd.read_csv(data_path)

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.sample(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data["HIV_active"].value_counts()

Dataset is highly inbalance

In [ ]:
!pip install rdkit
from rdkit import Chem
from rdkit.Chem import Draw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 34.5 MB/s eta 0:00:00


In [ ]:
samples = data['smiles'].sample(n=10, random_state=42).values
mols = [Chem.MolFromSmiles(smile) for smile in samples]
Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(200, 200))

In [ ]:
mols[0]

In [ ]:
dir(mols[0])

In [ ]:
import pandas as pd
from rdkit import Chem
from tqdm import tqdm

df = pd.read_csv("data/raw/HIV.csv").reset_index(drop=True)

invalid_indices = []
for index, row in tqdm(df.iterrows(), total=len(df)):
    mol_obj = Chem.MolFromSmiles(row["smiles"])
    if mol_obj is None:
        invalid_indices.append(index)

print(f"Invalid indices: {invalid_indices}")

clean_df = df.drop(invalid_indices).reset_index(drop=True)

clean_df.to_csv("data/raw/HIV_cleaned.csv", index=False)

print(f"Cleaned CSV saved: {len(clean_df)} rows (original: {len(df)}, removed: {len(invalid_indices)})")

 75%|███████▍  | 30706/41127 [00:10<00:05, 1969.02it/s][09:29:06] Explicit valence for atom # 12 Al, 7, is greater than permitted
[09:29:06] Explicit valence for atom # 13 Al, 7, is greater than permitted
 85%|████████▌ | 34971/41127 [00:11<00:01, 3149.17it/s][09:29:08] WARNING: not removing hydrogen atom without neighbors
[09:29:08] WARNING: not removing hydrogen atom without neighbors
100%|██████████| 41127/41127 [00:13<00:00, 2987.63it/s]


Invalid indices: [137, 987, 12882, 18293, 30784, 30785, 35728]
Cleaned CSV saved: 41120 rows (original: 41127, removed: 7)


In [ ]:


df = pd.read_csv("data/raw/HIV_cleaned.csv").reset_index(drop=True)

for index, row in tqdm(df.iterrows(), total=len(df)):
    mol_obj = Chem.MolFromSmiles(row["smiles"])
    if mol_obj is None:
        print("error...")

 85%|████████▍ | 34916/41120 [00:11<00:01, 3174.26it/s][09:30:54] WARNING: not removing hydrogen atom without neighbors
[09:30:54] WARNING: not removing hydrogen atom without neighbors
100%|██████████| 41120/41120 [00:13<00:00, 3004.26it/s]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/HIV_cleaned.csv").reset_index(drop=True)

train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["HIV_active"],   # keeps distribution same
    random_state=42
)

train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_val_df.to_csv("data/raw/HIV_train_val.csv", index=False)
test_df.to_csv("data/raw/HIV_test.csv", index=False)

print("Original distribution:")
print(df["HIV_active"].value_counts(normalize=True))

print("\nTrain/Val distribution:")
print(train_val_df["HIV_active"].value_counts(normalize=True))

print("\nTest distribution:")
print(test_df["HIV_active"].value_counts(normalize=True))

Original distribution:
HIV_active
0    0.964908
1    0.035092
Name: proportion, dtype: float64

Train/Val distribution:
HIV_active
0    0.964899
1    0.035101
Name: proportion, dtype: float64

Test distribution:
HIV_active
0    0.964981
1    0.035019
Name: proportion, dtype: float64


In [ ]:
!pip install deepchem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 11.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
data_path = 'data/raw/HIV.csv'
data = pd.read_csv(data_path)

data.iloc[0]["smiles"]

'CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)=[O+]2'

In [ ]:
import deepchem as dc

featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)

f = featurizer.featurize([data.iloc[0]["smiles"]])
f

array([<deepchem.feat.graph_data.GraphData object at 0x794e191b5ac0>],
      dtype=object)

In [ ]:
f.node_features

AttributeError: 'numpy.ndarray' object has no attribute 'node_features'